선형 모델 (linear, ridge, lasso, elasticnet)

# **데이터 전처리**

In [3]:
import pandas as pd

df=pd.read_csv('job_clean.csv')
df.head()

,job_title,job_category,salary_in_usd,experience_level,employment_type,work_setting,company_location,company_size,job_group,companyLoc_group
0,Data DevOps Engineer,Data Engineering,95012,Mid-level,Full-time,Hybrid,Germany,L,2,etc
1,Data Architect,Data Architecture and Modeling,186000,Senior,Full-time,In-person,United States,M,2,United States
2,Data Architect,Data Architecture and Modeling,81800,Senior,Full-time,In-person,United States,M,2,United States
3,Data Scientist,Data Science and Research,212000,Senior,Full-time,In-person,United States,M,2,United States
4,Data Scientist,Data Science and Research,93300,Senior,Full-time,In-person,United States,M,2,United States


# 원-핫 인코딩

In [4]:
print('get_dummies() 수행 전 데이터 Shape:', df.shape)
cols=["job_title", "job_category", "employment_type", "work_setting", "company_location","experience_level", "company_size","job_group","companyLoc_group"]
df_ohe=pd.get_dummies(df, columns=cols, drop_first=True)
print('get_dummies() 수행 후 데이터 Shape:', df_ohe.shape)

get_dummies() 수행 전 데이터 Shape: (7453, 10)
get_dummies() 수행 후 데이터 Shape: (7453, 182)


# **선형 모델 학습/예측/평가**

# 타깃 설정 및 데이터 분할

In [5]:
from sklearn.model_selection import train_test_split

y_target=df_ohe['salary_in_usd']
X_features=df_ohe.drop('salary_in_usd',axis=1,inplace=False)
X_train,X_test,y_train,y_test=train_test_split(X_features,y_target,
                                               test_size=0.2,random_state=42)

# LinearRegression

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_reg=LinearRegression()
lr_reg.fit(X_train, y_train)

y_pred_lr=lr_reg.predict(X_test)
lr_mse=mean_squared_error(y_test, y_pred_lr)
lr_rmse=np.sqrt(lr_mse)
lr_mae=mean_absolute_error(y_test, y_pred_lr)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(lr_mse, lr_rmse, lr_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_lr)))

MSE: 2540296788.552, RMSE: 50401.357, MAE: 39295.708
Variance Score: 0.345


**LinearRegression 최종 값:**

MSE: 2540296788.552

RMSE: 50401.357

MAE: 39295.708

R2: 0.345

@@ LinearRegression은 조정할 하이퍼 파라미터가 없기 때문!

# Ridge

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

r_reg=Ridge(alpha=10)
r_reg.fit(X_train, y_train)

y_pred_r=r_reg.predict(X_test)
r_mse=mean_squared_error(y_test, y_pred_r)
r_rmse=np.sqrt(r_mse)
r_mae=mean_absolute_error(y_test, y_pred_r)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(r_mse, r_rmse, r_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_r)))

MSE: 2542396163.567, RMSE: 50422.179, MAE: 39337.397
Variance Score: 0.345


**GridSearchCV 이용**

In [8]:
from sklearn.model_selection import GridSearchCV

param={'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

r_reg=Ridge()
grid_cv=GridSearchCV(r_reg, param, cv=10, scoring='r2', n_jobs=-1)
grid_cv.fit(X_train, y_train)

print('최적의 알파 값: {0:.4f}'.format(grid_cv.best_params_['alpha']))
print('최적의 R2: {0:.4f}'.format(grid_cv.best_score_))

최적의 알파 값: 10.0000
최적의 R2: 0.3289


- 음... 처음보다 결과가 안좋아짐.. 과적합?

**RidgeCV 이용**

In [9]:
from sklearn.linear_model import RidgeCV

ridge_cv=RidgeCV(alphas=np.logspace(-5, 2, 50), cv=10)
ridge_cv.fit(X_train, y_train)

print(f"최적의 알파 값: {ridge_cv.alpha_}")
print(f"최적의 R2: {ridge_cv.score(X_test, y_test):.3f}")


최적의 알파 값: 3.727593720314938
최적의 R2: 0.349


- R2 값이 1에 더 가까워짐 굿굿

In [16]:
r_reg=Ridge(alpha=3.727593720314938)
r_reg.fit(X_train, y_train)

y_pred_r=r_reg.predict(X_test)
r_mse=mean_squared_error(y_test, y_pred_r)
r_rmse=np.sqrt(r_mse)
r_mae=mean_absolute_error(y_test, y_pred_r)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(r_mse, r_rmse, r_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_r)))

MSE: 2527669166.916, RMSE: 50275.930, MAE: 39196.164
Variance Score: 0.349


**Ridge 최종 값:**

MSE: 2527669166.916

RMSE: 50275.930

MAE: 39196.164

R2: 0.349

# Lasso

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

l_reg=Lasso()
l_reg.fit(X_train, y_train)

y_pred_l=l_reg.predict(X_test)
l_mse=mean_squared_error(y_test, y_pred_l)
l_rmse=np.sqrt(l_mse)
l_mae=mean_absolute_error(y_test, y_pred_l)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(l_mse, l_rmse, l_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_l)))

MSE: 2530413525.855, RMSE: 50303.216, MAE: 39191.110
Variance Score: 0.348


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.118e+11, tolerance: 2.381e+09
  model = cd_fast.enet_coordinate_descent(


**LassoCV 이용**

In [18]:
from sklearn.linear_model import LassoCV

lasso_cv=LassoCV(alphas=np.logspace(-5, 1, 50), cv=5, n_jobs=-1)
lasso_cv.fit(X_train, y_train)

print(f"최적의 알파 값: {lasso_cv.alpha_}")
print(f"최적의 R2: {lasso_cv.score(X_test, y_test):.3f}")

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 19015714897.56836, tolerance: 1908618245.3414025
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 4771674577.65625, tolerance: 1908618245.3414025
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 35507098840.64844, tolerance: 1918766662.775063
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWa

최적의 알파 값: 10.0
최적의 R2: 0.349


- R2 값이 1에 더 가까워짐 굿굿

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

l_reg=Lasso(alpha=10)
l_reg.fit(X_train, y_train)

y_pred_l=l_reg.predict(X_test)
l_mse=mean_squared_error(y_test, y_pred_l)
l_rmse=np.sqrt(l_mse)
l_mae=mean_absolute_error(y_test, y_pred_l)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(l_mse, l_rmse, l_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_l)))

MSE: 2525762437.357, RMSE: 50256.964, MAE: 39131.135
Variance Score: 0.349


**Lasso 최종 값:**

MSE: 2525762437.357

RMSE: 50256.964

MAE: 39131.135

R2: 0.349

# ElasticNet

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

en_reg=ElasticNet(alpha=0.01, l1_ratio=0.5)
en_reg.fit(X_train, y_train)

y_pred_en=en_reg.predict(X_test)
en_mse=mean_squared_error(y_test, y_pred_en)
en_rmse=np.sqrt(en_mse)
en_mae=mean_absolute_error(y_test, y_pred_en)

print('MSE: {0:.3f}, RMSE: {1:.3F}, MAE: {2:.3F}'.format(en_mse, en_rmse, en_mae))
print('Variance Score: {0:.3f}'.format(r2_score(y_test, y_pred_en)))

MSE: 2583009725.302, RMSE: 50823.319, MAE: 39687.590
Variance Score: 0.334


**GridSearchCV 이용**

In [20]:
from sklearn.model_selection import GridSearchCV

params={
    'alpha': np.logspace(-4, 1, 50),
    'l1_ratio': np.linspace(0.1, 1, 10),
    'max_iter': [5000]
}

grid_cv=GridSearchCV(ElasticNet(), params, cv=5, n_jobs=-1)
grid_cv.fit(X_train, y_train)

print(f"Best Params: {grid_cv.best_params_}")
print(f"최적의 R2: {grid_cv.best_score_:.3f}")

Best Params: {'alpha': 0.0021209508879201904, 'l1_ratio': 0.7000000000000001, 'max_iter': 5000}
최적의 R2: 0.331


- 결과가 더 안좋아짐 ㅠㅠ

**RandomizedSearchCV 이용**

In [14]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

param_dist={
    'alpha': uniform(0.0001, 10),
    'l1_ratio': uniform(0.1, 0.9),
}

elastic_random = RandomizedSearchCV(ElasticNet(), param_dist, cv=10, n_iter=30, n_jobs=-1, random_state=42)
elastic_random.fit(X_train, y_train)

print(f"Best Params: {elastic_random.best_params_}")
print(f"최적의 R2: {elastic_random.best_score_:.3f}")

Best Params: {'alpha': 0.20594494295802446, 'l1_ratio': 0.9729188669457949}
최적의 R2: 0.320


- 역시나 결과가 더 안좋아짐.. 튜닝을 하지 않는 것이 낫다고 판단

**ElasticNet 최종값:**

MSE: 2583009725.302

RMSE: 50823.319

MAE: 39687.590

R2: 0.334

Ridge랑 Lasso가 그나마 결과가 좋음!